# Feasibility checkpoint

Khizar Mohammad Hashmi (26I-8009), Alishba Abbas (26I-8032), MDS-1A.

Shows that the data loads and is usable, the baseline runs, and the planned evaluation can be computed.

In [1]:
import warnings
warnings.filterwarnings('ignore')
import os, time
os.environ.setdefault('OMP_NUM_THREADS', '4')

'4'

In [2]:
SPLIT_SEED = 2026
HOLDOUT_SHARE = 0.20

K_RANGE = list(range(2, 13))
K_VARIETIES = 7  # number of bean varieties, used only to compare pipelines at the same k

N_INIT = 10  # restarts for every "proper" k-means fit
N_RESTARTS = 30  # single-start runs for the initialization check
N_BOOTSTRAP = 30  # refits on resampled data
FIT_SEED = 0

PCA_VARIANCE = 0.95

GAP_REFERENCES = 20
GAP_N_INIT = 3

CI_RESAMPLES = 2000
CI_SEED = 7
NULL_SEED = 11

SENSITIVITY_SPLIT_SEEDS = list(range(100, 110))

DATA_URL = "https://archive.ics.uci.edu/static/public/602/dry+bean+dataset.zip"
XLSX_SHA256 = None  # filled in data.py from the file we ship

import types
cfg = types.SimpleNamespace(**{k: v for k, v in globals().items() if k.isupper()})

## 1. The data loads

UCI Dry Bean, DOI 10.24432/C50S4B, licence CC BY 4.0.

In [3]:
import io
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split


LOCAL_FILE = next((p for p in (Path("data/Dry_Bean_Dataset.xlsx"), Path("../data/Dry_Bean_Dataset.xlsx")) if p.exists()), Path("data/Dry_Bean_Dataset.xlsx"))


def read_table(path=LOCAL_FILE):
    """Read our copy if we have it, otherwise download it from UCI."""
    path = Path(path)
    if path.exists():
        return pd.read_excel(path)
    print("Local copy not found, downloading from UCI (can be slow)...")
    with urllib.request.urlopen(DATA_URL, timeout=600) as reply:
        archive = zipfile.ZipFile(io.BytesIO(reply.read()))
    name = next(n for n in archive.namelist() if n.endswith(".xlsx"))
    return pd.read_excel(archive.open(name))


def features_and_labels(table):
    """Labels are kept apart from the features from this point on."""
    labels = table["Class"].astype(str).to_numpy()
    features = table.drop(columns="Class")
    return features.to_numpy(dtype=float), labels, list(features.columns)


def split(X, labels, seed=SPLIT_SEED):
    """Plain random split. We do not stratify, because that would use the labels."""
    return train_test_split(X, labels, test_size=HOLDOUT_SHARE, random_state=seed, shuffle=True)


def describe(table):
    return {
        "rows": int(table.shape[0]),
        "features": int(table.shape[1] - 1),
        "missing": int(table.isna().sum().sum()),
        "duplicate_rows": int(table.duplicated().sum()),
        "varieties": int(table["Class"].nunique()),
    }


def drop_duplicates(X, labels):
    _, keep = np.unique(X, axis=0, return_index=True)
    keep = np.sort(keep)
    return X[keep], labels[keep]

table = read_table()
print(describe(table))
X, labels, names = features_and_labels(table)
X_fit, X_hold, y_fit, y_hold = split(X, labels)
print(X_fit.shape, X_hold.shape)
table.head()

{'rows': 13611, 'features': 16, 'missing': 0, 'duplicate_rows': 68, 'varieties': 7}
(10888, 16) (2723, 16)


,Area,Perimeter,MajorAxisLength,MinorAxisLength,AspectRation,Eccentricity,ConvexArea,EquivDiameter,Extent,Solidity,roundness,Compactness,ShapeFactor1,ShapeFactor2,ShapeFactor3,ShapeFactor4,Class
0,28395,610.291,208.178117,173.888747,1.197191,0.549812,28715,190.141097,0.763923,0.988856,0.958027,0.913358,0.007332,0.003147,0.834222,0.998724,SEKER
1,28734,638.018,200.524796,182.734419,1.097356,0.411785,29172,191.272750,0.783968,0.984986,0.887034,0.953861,0.006979,0.003564,0.909851,0.998430,SEKER
2,29380,624.110,212.826130,175.931143,1.209713,0.562727,29690,193.410904,0.778113,0.989559,0.947849,0.908774,0.007244,0.003048,0.825871,0.999066,SEKER
3,30008,645.884,210.557999,182.516516,1.153638,0.498616,30724,195.467062,0.782681,0.976696,0.903936,0.928329,0.007017,0.003215,0.861794,0.994199,SEKER
4,30140,620.134,201.847882,190.279279,1.060798,0.333680,30417,195.896503,0.773098,0.990893,0.984877,0.970516,0.006697,0.003665,0.941900,0.999166,SEKER


## 2. Why scaling should matter

In [4]:
print(pd.Series(X_fit.std(axis=0, ddof=1), index=names).sort_values(ascending=False).round(4))

ConvexArea         29225.5366
Area               28787.5250
Perimeter            211.9081
MajorAxisLength       84.8564
EquivDiameter         58.4239
MinorAxisLength       44.3473
AspectRation           0.2474
ShapeFactor3           0.0992
Eccentricity           0.0921
Compactness            0.0618
roundness              0.0597
Extent                 0.0492
Solidity               0.0047
ShapeFactor4           0.0044
ShapeFactor1           0.0011
ShapeFactor2           0.0006
dtype: float64


## 3. The baseline runs: k-means on raw features

In [5]:
from dataclasses import dataclass

import numpy as np
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler



@dataclass(frozen=True)
class Recipe:
    name: str
    scaler: str = "none"  # none, zscore or minmax
    pca: float | None = None  # share of variance to keep, or None


RAW = Recipe("Raw")
RAW_PCA = Recipe("Raw+PCA", pca=PCA_VARIANCE)
Z = Recipe("Z", scaler="zscore")
Z_PCA = Recipe("Z+PCA", scaler="zscore", pca=PCA_VARIANCE)
RECIPES = [RAW, RAW_PCA, Z, Z_PCA]

# Each contrast changes exactly one factor.
CONTRASTS = [("Z", "Raw"), ("Z+PCA", "Raw+PCA"), ("Raw+PCA", "Raw"), ("Z+PCA", "Z")]


def make_preprocessor(recipe):
    steps = []
    if recipe.scaler == "zscore":
        steps.append(("scale", StandardScaler()))
    elif recipe.scaler == "minmax":
        steps.append(("scale", MinMaxScaler()))
    if recipe.pca is not None:
        steps.append(("pca", PCA(n_components=recipe.pca, svd_solver="full")))
    if not steps:
        steps.append(("identity", "passthrough"))
    return Pipeline(steps)


def transform_pair(recipe, X_fit, X_hold):
    """Fit on the 80% only, then apply to both parts. This is the leakage guard."""
    prep = make_preprocessor(recipe).fit(X_fit)
    return prep.transform(X_fit), prep.transform(X_hold), prep


def kmeans(k, seed=FIT_SEED, n_init=N_INIT):
    return KMeans(n_clusters=k, init="k-means++", n_init=n_init, random_state=seed)


def rotation_check(X_fit, k=7, seed=1):
    """Full-rank PCA only centres and rotates, so J must not move.

    Both runs start from the same centroids, one set rotated, so any gap
    in J is numerical error and not a different local optimum.
    """
    pca = PCA(svd_solver="full").fit(X_fit)
    rotated = pca.transform(X_fit)
    start = X_fit[np.random.default_rng(seed).choice(len(X_fit), k, replace=False)]
    a = KMeans(k, init=start, n_init=1, max_iter=300, tol=0, algorithm="lloyd").fit(X_fit)
    b = KMeans(k, init=pca.transform(start), n_init=1, max_iter=300, tol=0, algorithm="lloyd").fit(rotated)
    return a.inertia_, b.inertia_, a.labels_, b.labels_

A, B, _ = transform_pair(RAW, X_fit, X_hold)
t = time.time()
baseline = kmeans(cfg.K_VARIETIES).fit(A)
print('fit seconds', round(time.time() - t, 2), '| J', baseline.inertia_)

fit seconds 0.13 | J 467777529177.53516


## 4. The planned evaluation can be computed

The silhouette is computed on the held-out 20%. Labels are used only for ARI, after clustering.

In [6]:
import numpy as np
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler



def common_space(X_fit, X_hold):
    """One shared z-scored space, so silhouettes can be compared across recipes."""
    scaler = StandardScaler().fit(X_fit)
    return scaler.transform(X_fit), scaler.transform(X_hold)


def score(recipe, A_fit, A_hold, Z_hold, labels_hold, k, seed=FIT_SEED):
    """Fit k-means in the recipe's own space and score the held-out beans."""
    model = kmeans(k, seed).fit(A_fit)
    pred = model.predict(A_hold)
    return {
        "recipe": recipe.name,
        "k": k,
        "J_fit": model.inertia_,
        "silhouette_own": silhouette_score(A_hold, pred),
        "silhouette_common": silhouette_score(Z_hold, pred),
        "ari": adjusted_rand_score(labels_hold, pred),
    }, pred


def ari_interval(labels, pred, seed=CI_SEED):
    """Percentile interval for ARI from resampling the held-out beans."""
    rng = np.random.default_rng(seed)
    n = len(labels)
    values = []
    for _ in range(CI_RESAMPLES):
        idx = rng.integers(0, n, n)
        values.append(adjusted_rand_score(labels[idx], pred[idx]))
    return np.percentile(values, [2.5, 97.5])


def ari_difference_interval(labels, pred_a, pred_b, seed=CI_SEED):
    """Paired interval for ARI(a) - ARI(b), resampling the same beans for both."""
    rng = np.random.default_rng(seed)
    n = len(labels)
    diffs = []
    for _ in range(CI_RESAMPLES):
        idx = rng.integers(0, n, n)
        diffs.append(adjusted_rand_score(labels[idx], pred_a[idx]) - adjusted_rand_score(labels[idx], pred_b[idx]))
    return np.percentile(diffs, [2.5, 97.5])


def shuffle_columns(X, seed):
    """Shuffle every feature on its own: same marginals, no relation between features."""
    return np.column_stack([np.random.default_rng(seed + j).permutation(X[:, j]) for j in range(X.shape[1])])


def null_silhouette(recipe, X_fit, X_hold, k, seed=NULL_SEED):
    """Run the recipe on data with no real structure and score it the same way."""
    fit, hold = shuffle_columns(X_fit, seed), shuffle_columns(X_hold, seed + 1000)
    prep = make_preprocessor(recipe).fit(fit)
    A_fit, A_hold = prep.transform(fit), prep.transform(hold)
    pred = kmeans(k).fit(A_fit).predict(A_hold)
    return silhouette_score(A_hold, pred)

Z_fit, Z_hold = common_space(X_fit, X_hold)
result, pred = score(RAW, A, B, Z_hold, y_hold, cfg.K_VARIETIES)
print(result)
print('ARI 95% interval', ari_interval(y_hold, pred).round(3))

{'recipe': 'Raw', 'k': 7, 'J_fit': 467777529177.53516, 'silhouette_own': 0.5291735620700861, 'silhouette_common': 0.08364245209618813, 'ari': 0.387437568947455}


ARI 95% interval [0.366 0.411]


## 5. One stability number

In [7]:
from itertools import combinations

import numpy as np
from sklearn.metrics import adjusted_rand_score



def _pairwise_ari(predictions):
    return np.array([adjusted_rand_score(a, b) for a, b in combinations(predictions, 2)])


def _summary(recipe_name, check, values, extra=None):
    row = {
        "recipe": recipe_name,
        "check": check,
        "mean_ari": values.mean(),
        "sd_ari": values.std(ddof=1),
        "min_ari": values.min(),
        "share_pairs_above_0_9": float((values >= 0.9).mean()),
    }
    row.update(extra or {})
    return row


def restart_stability(recipe_name, A_fit, A_hold, k=K_VARIETIES):
    """Single-start k-means, different seed each time, same data."""
    models = [kmeans(k, seed=s, n_init=1).fit(A_fit) for s in range(N_RESTARTS)]
    predictions = [m.predict(A_hold) for m in models]
    J = np.array([m.inertia_ for m in models])
    return _summary(recipe_name, "restarts", _pairwise_ari(predictions),
                    {"worst_J_over_best_J": J.max() / J.min()})


def bootstrap_stability(recipe, X_fit, X_hold, k=K_VARIETIES):
    """Resample the fit part and refit the whole recipe, scaler and PCA included."""
    predictions = []
    components = []
    for s in range(N_BOOTSTRAP):
        idx = np.random.default_rng(s).integers(0, len(X_fit), len(X_fit))
        prep = make_preprocessor(recipe).fit(X_fit[idx])
        model = kmeans(k, seed=s).fit(prep.transform(X_fit[idx]))
        predictions.append(model.predict(prep.transform(X_hold)))
        if recipe.pca is not None:
            components.append(prep.named_steps["pca"].n_components_)
    extra = {"components_seen": sorted(set(int(c) for c in components))} if components else {}
    return _summary(recipe.name, "bootstrap", _pairwise_ari(predictions), extra)

print(restart_stability('Raw', A, B))

{'recipe': 'Raw', 'check': 'restarts', 'mean_ari': np.float64(0.8187474584254787), 'sd_ari': np.float64(0.18485234106907014), 'min_ari': np.float64(0.49634142831516553), 'share_pairs_above_0_9': 0.37471264367816093, 'worst_J_over_best_J': np.float64(1.3366813955515484)}
